# 04 — Identifiability Analysis

This notebook tests whether the fitted parameters are uniquely determined.

It builds RMSE landscapes around the fitted values:
- 1D sweeps for `R_hidden`, `L_eff`, and `C_eff`
- practical 5% and 10% RMSE uncertainty intervals
- 2D heatmaps showing parameter tradeoffs
- reconstruction trend figures with uncertainty bars

This is the inverse-problem core of the project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import curve_fit

DATA_PATH = Path("../data/processed/resistance_sweep_combined_clean.csv")

PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../figures/identifiability")
GRID_1D_DIR = PROCESSED_DIR / "identifiability_1d"
GRID_2D_DIR = PROCESSED_DIR / "identifiability_2d"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
GRID_1D_DIR.mkdir(parents=True, exist_ok=True)
GRID_2D_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 12,
    "axes.grid": True,
})

In [ ]:
df = pd.read_csv(DATA_PATH)

required = ["Resistance", "Frequency_Hz", "Gain"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Current columns: {df.columns.tolist()}")

for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required)
df = df.sort_values(["Resistance", "Frequency_Hz"]).reset_index(drop=True)

df.head()

In [ ]:
def gain_model(f, R_measured, R_hidden, L, C):
    omega = 2 * np.pi * f
    R_total = R_measured + R_hidden
    X = omega * L - 1 / (omega * C)
    return R_measured / np.sqrt(R_total**2 + X**2)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def resonance_frequency(L, C):
    return 1 / (2 * np.pi * np.sqrt(L * C))

In [ ]:
BOUNDS_LOWER = [0, 5e-3, 100e-9]
BOUNDS_UPPER = [200, 20e-3, 400e-9]
INITIAL_GUESS = [40, 10e-3, 220e-9]

def fit_resistance_curve(group):
    R_measured = group["Resistance"].iloc[0]
    f = group["Frequency_Hz"].values
    y = group["Gain"].values

    def model_for_fit(f, R_hidden, L, C):
        return gain_model(f, R_measured, R_hidden, L, C)

    try:
        popt, pcov = curve_fit(
            model_for_fit,
            f,
            y,
            p0=INITIAL_GUESS,
            bounds=(BOUNDS_LOWER, BOUNDS_UPPER),
            maxfev=50000
        )

        R_hidden, L, C = popt
        y_fit = model_for_fit(f, R_hidden, L, C)

        return {
            "Resistance": R_measured,
            "R_hidden_Ohm": R_hidden,
            "L_H": L,
            "L_mH": L * 1e3,
            "C_F": C,
            "C_nF": C * 1e9,
            "f0_Hz": resonance_frequency(L, C),
            "RMSE": rmse(y, y_fit),
            "Fit_Success": True,
            "Error": ""
        }

    except Exception as e:
        return {
            "Resistance": R_measured,
            "R_hidden_Ohm": np.nan,
            "L_H": np.nan,
            "L_mH": np.nan,
            "C_F": np.nan,
            "C_nF": np.nan,
            "f0_Hz": np.nan,
            "RMSE": np.nan,
            "Fit_Success": False,
            "Error": str(e)
        }

In [ ]:
fit_rows = []

for R, group in df.groupby("Resistance"):
    fit_rows.append(fit_resistance_curve(group))

fits = pd.DataFrame(fit_rows).sort_values("Resistance")
fits.to_csv(PROCESSED_DIR / "parameter_reconstruction_identifiability_baseline.csv", index=False)

fits

## 1D identifiability sweeps

A narrow RMSE minimum means the parameter is strongly constrained.  
A broad minimum means many values produce almost the same curve.

In [ ]:
def uncertainty_range(grid, errors, factor):
    min_error = np.nanmin(errors)
    mask = errors <= factor * min_error

    if not np.any(mask):
        return np.nan, np.nan

    return grid[mask].min(), grid[mask].max()

def compute_1d_sweeps_for_resistance(R):
    group = df[df["Resistance"] == R].sort_values("Frequency_Hz")
    fit = fits[fits["Resistance"] == R].iloc[0]

    if not fit["Fit_Success"]:
        return None

    f = group["Frequency_Hz"].values
    y = group["Gain"].values

    R_best = fit["R_hidden_Ohm"]
    L_best = fit["L_H"]
    C_best = fit["C_F"]

    R_grid = np.linspace(BOUNDS_LOWER[0], BOUNDS_UPPER[0], 500)
    L_grid = np.linspace(BOUNDS_LOWER[1], BOUNDS_UPPER[1], 500)
    C_grid = np.linspace(BOUNDS_LOWER[2], BOUNDS_UPPER[2], 500)

    rmse_R = np.array([rmse(y, gain_model(f, R, R_hidden, L_best, C_best)) for R_hidden in R_grid])
    rmse_L = np.array([rmse(y, gain_model(f, R, R_best, L, C_best)) for L in L_grid])
    rmse_C = np.array([rmse(y, gain_model(f, R, R_best, L_best, C)) for C in C_grid])

    sweep_df = pd.DataFrame({
        "R_hidden_Ohm": R_grid,
        "RMSE_R_hidden": rmse_R,
        "L_H": L_grid,
        "L_mH": L_grid * 1e3,
        "RMSE_L": rmse_L,
        "C_F": C_grid,
        "C_nF": C_grid * 1e9,
        "RMSE_C": rmse_C,
    })

    sweep_df.to_csv(GRID_1D_DIR / f"identifiability_1d_R_{int(R)}ohm.csv", index=False)
    return sweep_df

In [ ]:
uncertainty_rows = []

for R in sorted(df["Resistance"].unique()):
    sweep = compute_1d_sweeps_for_resistance(R)

    if sweep is None:
        continue

    fit = fits[fits["Resistance"] == R].iloc[0]

    R5_low, R5_high = uncertainty_range(sweep["R_hidden_Ohm"].values, sweep["RMSE_R_hidden"].values, 1.05)
    R10_low, R10_high = uncertainty_range(sweep["R_hidden_Ohm"].values, sweep["RMSE_R_hidden"].values, 1.10)

    L5_low, L5_high = uncertainty_range(sweep["L_mH"].values, sweep["RMSE_L"].values, 1.05)
    L10_low, L10_high = uncertainty_range(sweep["L_mH"].values, sweep["RMSE_L"].values, 1.10)

    C5_low, C5_high = uncertainty_range(sweep["C_nF"].values, sweep["RMSE_C"].values, 1.05)
    C10_low, C10_high = uncertainty_range(sweep["C_nF"].values, sweep["RMSE_C"].values, 1.10)

    uncertainty_rows.append({
        "Resistance": R,
        "Best_R_hidden_Ohm": fit["R_hidden_Ohm"],
        "R_hidden_5pct_low": R5_low,
        "R_hidden_5pct_high": R5_high,
        "R_hidden_10pct_low": R10_low,
        "R_hidden_10pct_high": R10_high,
        "Best_L_mH": fit["L_mH"],
        "L_5pct_low_mH": L5_low,
        "L_5pct_high_mH": L5_high,
        "L_10pct_low_mH": L10_low,
        "L_10pct_high_mH": L10_high,
        "Best_C_nF": fit["C_nF"],
        "C_5pct_low_nF": C5_low,
        "C_5pct_high_nF": C5_high,
        "C_10pct_low_nF": C10_low,
        "C_10pct_high_nF": C10_high,
        "Min_RMSE": fit["RMSE"],
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows)
uncertainty_summary.to_csv(PROCESSED_DIR / "identifiability_uncertainty_summary.csv", index=False)

uncertainty_summary

In [ ]:
def plot_1d_sweep(x, y, best_x, xlabel, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(x, y, linewidth=2)
    ax.axvline(best_x, linestyle="--", label="Best fit")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("RMSE")
    ax.set_title(title)
    ax.grid(True, alpha=0.35)
    ax.legend()

    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

for R in sorted(df["Resistance"].unique()):
    sweep_path = GRID_1D_DIR / f"identifiability_1d_R_{int(R)}ohm.csv"

    if not sweep_path.exists():
        continue

    sweep = pd.read_csv(sweep_path)
    fit = fits[fits["Resistance"] == R].iloc[0]

    plot_1d_sweep(
        sweep["R_hidden_Ohm"], sweep["RMSE_R_hidden"], fit["R_hidden_Ohm"],
        "R_hidden (Ω)", f"R_hidden Identifiability, R = {R:.0f} Ω",
        f"R_hidden_identifiability_R_{int(R)}ohm.png"
    )

    plot_1d_sweep(
        sweep["L_mH"], sweep["RMSE_L"], fit["L_mH"],
        "L_eff (mH)", f"L Identifiability, R = {R:.0f} Ω",
        f"L_identifiability_R_{int(R)}ohm.png"
    )

    plot_1d_sweep(
        sweep["C_nF"], sweep["RMSE_C"], fit["C_nF"],
        "C_eff (nF)", f"C Identifiability, R = {R:.0f} Ω",
        f"C_identifiability_R_{int(R)}ohm.png"
    )

## 2D identifiability maps

The most important tradeoff is often L-C because the resonance frequency depends mainly on the product `LC`.

Long valleys mean parameter non-uniqueness.

In [ ]:
def compute_2d_maps(R, grid_size=120):
    group = df[df["Resistance"] == R].sort_values("Frequency_Hz")
    fit = fits[fits["Resistance"] == R].iloc[0]

    if not fit["Fit_Success"]:
        return None

    f = group["Frequency_Hz"].values
    y = group["Gain"].values

    R_best = fit["R_hidden_Ohm"]
    L_best = fit["L_H"]
    C_best = fit["C_F"]

    R_grid = np.linspace(BOUNDS_LOWER[0], BOUNDS_UPPER[0], grid_size)
    L_grid = np.linspace(BOUNDS_LOWER[1], BOUNDS_UPPER[1], grid_size)
    C_grid = np.linspace(BOUNDS_LOWER[2], BOUNDS_UPPER[2], grid_size)

    rmse_LC = np.zeros((grid_size, grid_size))
    rmse_RL = np.zeros((grid_size, grid_size))
    rmse_RC = np.zeros((grid_size, grid_size))

    for i, L in enumerate(L_grid):
        for j, C in enumerate(C_grid):
            rmse_LC[i, j] = rmse(y, gain_model(f, R, R_best, L, C))

    for i, R_hidden in enumerate(R_grid):
        for j, L in enumerate(L_grid):
            rmse_RL[i, j] = rmse(y, gain_model(f, R, R_hidden, L, C_best))

    for i, R_hidden in enumerate(R_grid):
        for j, C in enumerate(C_grid):
            rmse_RC[i, j] = rmse(y, gain_model(f, R, R_hidden, L_best, C))

    pd.DataFrame(rmse_LC, index=L_grid * 1e3, columns=C_grid * 1e9).to_csv(
        GRID_2D_DIR / f"LC_rmse_grid_R_{int(R)}ohm.csv"
    )
    pd.DataFrame(rmse_RL, index=R_grid, columns=L_grid * 1e3).to_csv(
        GRID_2D_DIR / f"Rhidden_L_rmse_grid_R_{int(R)}ohm.csv"
    )
    pd.DataFrame(rmse_RC, index=R_grid, columns=C_grid * 1e9).to_csv(
        GRID_2D_DIR / f"Rhidden_C_rmse_grid_R_{int(R)}ohm.csv"
    )

    return {
        "R_grid": R_grid,
        "L_grid": L_grid,
        "C_grid": C_grid,
        "rmse_LC": rmse_LC,
        "rmse_RL": rmse_RL,
        "rmse_RC": rmse_RC,
        "fit": fit
    }

In [ ]:
def plot_heatmap(x, y, z, best_x, best_y, xlabel, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.contourf(x, y, z, levels=50)
    ax.scatter(best_x, best_y, marker="x", s=120, label="Best fit")

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    fig.colorbar(im, ax=ax, label="RMSE")
    ax.legend()

    fig.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

selected_resistances = [47, 150, 470]
available_R = sorted(df["Resistance"].unique())
selected_resistances = [R for R in selected_resistances if R in available_R]

for R in selected_resistances:
    result = compute_2d_maps(R, grid_size=120)

    if result is None:
        continue

    fit = result["fit"]
    R_grid = result["R_grid"]
    L_grid = result["L_grid"]
    C_grid = result["C_grid"]

    plot_heatmap(
        C_grid * 1e9, L_grid * 1e3, result["rmse_LC"],
        fit["C_nF"], fit["L_mH"],
        "C_eff (nF)", "L_eff (mH)",
        f"L-C Identifiability Map, R = {R:.0f} Ω",
        f"LC_heatmap_R_{int(R)}ohm.png"
    )

    plot_heatmap(
        L_grid * 1e3, R_grid, result["rmse_RL"],
        fit["L_mH"], fit["R_hidden_Ohm"],
        "L_eff (mH)", "R_hidden (Ω)",
        f"R_hidden-L Identifiability Map, R = {R:.0f} Ω",
        f"Rhidden_L_heatmap_R_{int(R)}ohm.png"
    )

    plot_heatmap(
        C_grid * 1e9, R_grid, result["rmse_RC"],
        fit["C_nF"], fit["R_hidden_Ohm"],
        "C_eff (nF)", "R_hidden (Ω)",
        f"R_hidden-C Identifiability Map, R = {R:.0f} Ω",
        f"Rhidden_C_heatmap_R_{int(R)}ohm.png"
    )

## Reconstruction trend figures with uncertainty bars

These figures directly show whether recovered physical parameters are invariant across measured resistance.

In [ ]:
FIT_PATH = PROCESSED_DIR / "reconstruction_summary.csv"
fit = pd.read_csv(FIT_PATH)
unc = pd.read_csv(PROCESSED_DIR / "identifiability_uncertainty_summary.csv")

TREND_DIR = Path("../figures/reconstruction_trends")
TREND_DIR.mkdir(parents=True, exist_ok=True)

def save_errorbar_plot(
    x, y,
    yerr_low=None,
    yerr_high=None,
    xlabel="",
    ylabel="",
    title="",
    filename="figure"
):
    fig, ax = plt.subplots(figsize=(8, 5))

    if yerr_low is not None and yerr_high is not None:
        lower_err = np.maximum(y - yerr_low, 0)
        upper_err = np.maximum(yerr_high - y, 0)

        ax.errorbar(x, y, yerr=[lower_err, upper_err], fmt="o-", linewidth=2, capsize=4)
    else:
        ax.plot(x, y, "o-", linewidth=2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.35)

    fig.savefig(TREND_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(TREND_DIR / f"{filename}.svg", bbox_inches="tight")
    plt.show()

In [ ]:
save_errorbar_plot(
    x=unc["Resistance"],
    y=unc["Best_R_hidden_Ohm"],
    yerr_low=unc["R_hidden_10pct_low"],
    yerr_high=unc["R_hidden_10pct_high"],
    xlabel="Measured Resistance R (Ω)",
    ylabel="Recovered Hidden Resistance (Ω)",
    title="Recovered Hidden Resistance vs Measured Resistance",
    filename="R_hidden_vs_measured_R_with_uncertainty"
)

save_errorbar_plot(
    x=unc["Resistance"],
    y=unc["Best_L_mH"],
    yerr_low=unc["L_10pct_low_mH"],
    yerr_high=unc["L_10pct_high_mH"],
    xlabel="Measured Resistance R (Ω)",
    ylabel="Recovered Effective Inductance (mH)",
    title="Recovered Effective Inductance vs Measured Resistance",
    filename="L_eff_vs_measured_R_with_uncertainty"
)

save_errorbar_plot(
    x=unc["Resistance"],
    y=unc["Best_C_nF"],
    yerr_low=unc["C_10pct_low_nF"],
    yerr_high=unc["C_10pct_high_nF"],
    xlabel="Measured Resistance R (Ω)",
    ylabel="Recovered Effective Capacitance (nF)",
    title="Recovered Effective Capacitance vs Measured Resistance",
    filename="C_eff_vs_measured_R_with_uncertainty"
)

save_errorbar_plot(
    x=fit["Resistance"],
    y=fit["RMSE"],
    xlabel="Measured Resistance R (Ω)",
    ylabel="RMSE",
    title="Model Error vs Measured Resistance",
    filename="RMSE_vs_measured_R"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

scatter = ax.scatter(
    fit["C_eff_nF"],
    fit["L_eff_mH"],
    c=fit["Resistance"],
    s=90
)

for _, row in fit.iterrows():
    ax.annotate(
        f"{int(row['Resistance'])}Ω",
        (row["C_eff_nF"], row["L_eff_mH"]),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9
    )

ax.set_xlabel("Recovered Effective Capacitance (nF)")
ax.set_ylabel("Recovered Effective Inductance (mH)")
ax.set_title("L-C Tradeoff in Parameter Reconstruction")
ax.grid(True, alpha=0.35)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Measured Resistance R (Ω)")

fig.savefig(TREND_DIR / "LC_tradeoff_scatter.png", dpi=300, bbox_inches="tight")
fig.savefig(TREND_DIR / "LC_tradeoff_scatter.svg", bbox_inches="tight")

plt.show()

In [ ]:
fit["LC_product"] = (fit["L_eff_mH"] * 1e-3) * (fit["C_eff_nF"] * 1e-9)
fit["f0_reconstructed_Hz"] = 1 / (2 * np.pi * np.sqrt(fit["LC_product"]))

f0_nominal = 1 / (2 * np.pi * np.sqrt(10e-3 * 220e-9))

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(fit["Resistance"], fit["f0_reconstructed_Hz"], "o-", linewidth=2)
ax.axhline(f0_nominal, linestyle="--", label="Nominal f0")

ax.set_xlabel("Measured Resistance R (Ω)")
ax.set_ylabel("Reconstructed Resonance Frequency (Hz)")
ax.set_title("Reconstructed Resonance Frequency vs Measured Resistance")
ax.grid(True, alpha=0.35)
ax.legend()

fig.savefig(TREND_DIR / "reconstructed_f0_vs_measured_R.png", dpi=300, bbox_inches="tight")
fig.savefig(TREND_DIR / "reconstructed_f0_vs_measured_R.svg", bbox_inches="tight")

plt.show()

fit.to_csv(PROCESSED_DIR / "reconstruction_trend_table.csv", index=False)
fit

## Final interpretation

A low RMSE means the curve was fitted well.

But if `L_eff`, `C_eff`, and `R_hidden` change across resistance values, the inverse problem is not uniquely reconstructing fixed physical components.

The recovered resonance frequency can remain nearly constant while L and C move in opposite directions. That is direct evidence of parameter tradeoff.

This is the central system-identification result.